# Experimento MAREA — Competidor: **MAREA**

**MAREA-base — Metaheuristic Algorithm for Regulated Exploration and Adaptation
Parámetros calibrados: N=80, α_max=0.99, β_max=0.3, γ_max=0.2**

### Protocolo experimental
- **Benchmarks:** 71 funciones CEC 2014 (30) + CEC 2017 (29) + CEC 2022 (12)
- **Dimensiones:** 10D y 50D (CEC2022: 10D y 20D)
- **MaxFES:** 5,000 / 50,000 / 500,000 / 5,000,000
- **Corridas:** 51 independientes por configuración
- **Guardado:** incremental en JSON — se puede interrumpir y reanudar

> Ejecutar las celdas en orden. Cada celda de experimento es independiente
> y guarda sus resultados automáticamente al terminar.


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Asegurar que el directorio del proyecto esté en el path
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from experimento_config import (
    run_experiment, load_partial, CEC_REGISTRY,
    MAXFES, DIMS, N_RUNS, POP_SIZE, RESULTS_DIR
)

ALG_NAME  = 'MAREA'
N_WORKERS = 8      # esta maquina tiene 10 cores y MAREA corre sola

# Guarda de seguridad: las celdas de 50k / 500k / 5M no se lanzan con "Run All".
# Ponlo en True solo cuando quieras comprometer horas de maquina.
RUN_HEAVY = False

# Cargar el algoritmo
from marea_benchmark import marea, MAREAParams
p = MAREAParams()

# La parada anticipada debe estar DESACTIVADA: el protocolo CEC exige agotar
# MaxFES. Con early_stop=True MAREA abandonaba hasta al 0.5% del presupuesto
# mientras los competidores gastaban el 100%.
assert p.early_stop is False, 'early_stop debe estar desactivado para el benchmark'

def alg_fn(f, bounds, N, T, seed):
    p.N = N; p.T = T
    return marea(f, bounds, p, seed=seed)

print(f'\u2713 {ALG_NAME} cargado correctamente')
print(f'  early_stop = {p.early_stop}  (agota MaxFES completo)')
print(f'  Resultados en: {RESULTS_DIR}/')
print(f'  Funciones CEC disponibles: {len(CEC_REGISTRY)}')
print(f'  Workers: {N_WORKERS} | RUN_HEAVY: {RUN_HEAVY}')


## Verificación rápida (smoke test)
Corre 1 función × 3 semillas para confirmar que todo funciona antes del experimento completo.

In [ ]:
from experimento_config import get_problems_for_dim
import time

problems_10d = get_problems_for_dim(10)
test_prob    = problems_10d[0]   # F1 de CEC2014
print(f'Probando con: {test_prob["name"]} | dim={test_prob["dim"]}')

T_TEST = 62   # 62 iteraciones x 80 individuos = 4,960 FES ~ MaxFES=5,000

t0 = time.time()
for seed in range(3):
    r = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=seed)
    ok = 'OK' if r['iters'] == T_TEST else 'PARADA ANTICIPADA'
    print(f'  seed={seed}  best_f={r["best_f"]:.4e}  iters={r["iters"]}/{T_TEST}  [{ok}]')
print(f'Tiempo: {time.time()-t0:.2f}s')

# Si alguna corrida no llega a T_TEST, la parada anticipada sigue activa.
assert all(alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST,
                  seed=s)['iters'] == T_TEST for s in range(3)), \
    'Alguna corrida paro antes de agotar el presupuesto'
print('\u2713 Presupuesto agotado en las 3 semillas. Listo para el experimento.')


## Experimento — MaxFES = 5,000  ← **ejecutar esta**
*Caso de uso: optimización costosa / tiempo-real*

Prueba local tras desactivar la parada anticipada. Coste medido:
**~13 min-core en D=10 y ~32 min-core en D=50** → unos **6 minutos**
de reloj con 8 workers.

Los JSON anteriores de MAREA se movieron a `resultados_marea_earlystop/`,
así que estas celdas recalculan desde cero (no reanudan).


In [ ]:
# ── MaxFES = 5,000 ──────────────────────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecución retoma desde donde quedó.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000, n_workers=N_WORKERS)


## Experimento — MaxFES = 50,000
*Caso de uso: aplicaciones prácticas estándar*


In [ ]:
# ── MaxFES = 50,000 ──────────────────────────────────────────────
# Coste estimado sin parada anticipada: ~17 min (D=10) + ~40 min (D=50) con 8 workers
# Requiere RUN_HEAVY = True en la primera celda.

if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=50000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~17 min (D=10) + ~40 min (D=50) con 8 workers')


## Experimento — MaxFES = 500,000
*Caso de uso: convención CEC estándar*


In [ ]:
# ── MaxFES = 500,000 ──────────────────────────────────────────────
# Coste estimado sin parada anticipada: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers
# Requiere RUN_HEAVY = True en la primera celda.

if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=500000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers')


## Experimento — MaxFES = 5,000,000
*Caso de uso: búsqueda exploratoria profunda*


In [ ]:
# ── MaxFES = 5,000,000 ──────────────────────────────────────────────
# Coste estimado sin parada anticipada: ~10x el de 500k — noche completa de maquina
# Requiere RUN_HEAVY = True en la primera celda.

if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~10x el de 500k — noche completa de maquina')


## Resumen de resultados
Muestra el ranking promedio de este algoritmo en cada configuración.

In [ ]:
import json

print(f'\n=== RESUMEN {ALG_NAME} ===')
print(f'{"Config":<30} {"Completadas":>12} {"Mean":>14} {"Best":>14}')
print('-'*72)

for mf in MAXFES:
    for nd in DIMS:
        data = load_partial(ALG_NAME, nd, mf)
        vals = list(data.values())
        if vals:
            label = f'D={nd:2d} MaxFES={mf:>10,}'
            print(f'{label:<30} {len(vals):>12} {np.mean(vals):>14.4e} {np.min(vals):>14.4e}')
        else:
            print(f'D={nd:2d} MaxFES={mf:>10,}  (sin resultados aún)')


## Ablación — efecto de la parada anticipada
Compara los resultados nuevos contra los de `resultados_marea_earlystop/`
(la variante que abandonaba antes de agotar MaxFES).


In [ ]:
import json, os, glob
import numpy as np

OLD_DIR = 'resultados_marea_earlystop'

def _load(path):
    if not os.path.exists(path):
        return None
    raw = json.load(open(path))
    M = np.full((71, N_RUNS), np.nan)
    for k, v in raw.items():
        a, b = k.split('_')
        M[int(a[3:]), int(b[3:])] = v
    return M

print(f'{"Config":<22}{"n":>7}{"media nueva":>15}{"media vieja":>15}{"mejora":>10}')
print('-' * 69)
for mf in MAXFES:
    for nd in DIMS:
        new = _load(os.path.join(RESULTS_DIR, f'{ALG_NAME}_D{nd}_FES{mf}.json'))
        old = _load(os.path.join(OLD_DIR,     f'{ALG_NAME}_D{nd}_FES{mf}.json'))
        if new is None or old is None:
            continue
        # Comparar solo las funciones completas en ambos
        ok = [i for i in range(71)
              if not np.isnan(new[i]).any() and not np.isnan(old[i]).any()]
        if not ok:
            continue
        mn = np.mean([np.mean(new[i]) for i in ok])
        mo = np.mean([np.mean(old[i]) for i in ok])
        # Cuantas funciones mejoran (media mas baja = mejor)
        better = sum(1 for i in ok if np.mean(new[i]) < np.mean(old[i]))
        label = f'D={nd:2d} FES={mf:,}'
        print(f'{label:<22}{len(ok):>7}{mn:>15.4e}{mo:>15.4e}'
              f'{better:>7}/{len(ok)}')
print('\nUltima columna: funciones donde la version sin parada anticipada es mejor.')
